In [0]:
print("=== CLIMA ===")
spark.sql("SELECT DISTINCT departamento FROM bronze.clima_raw ORDER BY departamento").show(truncate=False)

In [0]:
print("=== VENTAS ===")
spark.sql("SELECT DISTINCT departamento_destino FROM gold.resumen_ventas_mensual ORDER BY departamento_destino").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

ventas = spark.table("gold.resumen_ventas_mensual")

ventas_mes_depto = (
    ventas
    .withColumn("anio", F.year("mes"))
    .withColumn("mes_num", F.month("mes"))
    .groupBy("departamento_destino", "anio", "mes_num")
    .agg(F.round(F.sum("tm_vendidas"), 1).alias("tm_vendidas_total"))
)

ventas_mes_depto.orderBy("anio", "mes_num", "departamento_destino").show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

ventas_mes_depto = (
    spark.table("gold.resumen_ventas_mensual")
    .withColumn("anio", F.year("mes"))
    .withColumn("mes_num", F.month("mes"))
    .groupBy("departamento_destino", "anio", "mes_num")
    .agg(F.round(F.sum("tm_vendidas"), 1).alias("tm_vendidas_total"))
)

clima = (
    spark.table("bronze.clima_raw")
    .select("departamento", "anio", "mes", "precip_pronosticada_mm")
)

dataset = (
    ventas_mes_depto.join(
        clima,
        on=[
            ventas_mes_depto.departamento_destino == clima.departamento,
            ventas_mes_depto.anio == clima.anio,
            ventas_mes_depto.mes_num == clima.mes,
        ],
        how="inner"   
    )
    .select(
        ventas_mes_depto.departamento_destino,
        ventas_mes_depto.anio,
        ventas_mes_depto.mes_num,
        clima.precip_pronosticada_mm,   
        ventas_mes_depto.tm_vendidas_total,  
    )
)

print("Filas en el dataset final:", dataset.count())
dataset.orderBy("anio", "mes_num", "departamento_destino").show(12, truncate=False)

In [0]:
df = dataset.select("precip_pronosticada_mm", "tm_vendidas_total").toPandas()

print("Forma:", df.shape)

df.head()

In [0]:
from sklearn.model_selection import train_test_split

X = df[["precip_pronosticada_mm"]]

y = df["tm_vendidas_total"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      
    random_state=42     
)

print("Entrenamiento:", X_train.shape[0], "filas")
print("Prueba:", X_test.shape[0], "filas")

In [0]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()

modelo.fit(X_train, y_train)

print("Fórmula aprendida:")
print(f"  tm_vendidas = {modelo.coef_[0]:.2f} * lluvia + {modelo.intercept_:.2f}")

In [0]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = modelo.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²:  {r2:.3f}")
print(f"MAE: {mae:.1f} toneladas")

In [0]:

df = dataset.select(
    "departamento_destino",
    "precip_pronosticada_mm",
    "tm_vendidas_total"
).toPandas()

print("Forma:", df.shape)  
df.head()

In [0]:
import pandas as pd

df_encoded = pd.get_dummies(df, columns=["departamento_destino"], dtype=int)

print("Columnas ahora:", list(df_encoded.columns))
df_encoded.head()

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = df_encoded.drop(columns=["tm_vendidas_total"])
y = df_encoded["tm_vendidas_total"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo2 = LinearRegression()
modelo2.fit(X_train, y_train)

y_pred = modelo2.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²:  {r2:.3f}   (antes: 0.263)")
print(f"MAE: {mae:.1f} toneladas   (antes: 1446.8)")

In [0]:

datos = dataset.select(
    "precip_pronosticada_mm",
    "tm_vendidas_total"
).toPandas()

datos_real = dataset.join(
    spark.table("bronze.clima_raw").select(
        F.col("departamento").alias("dep_c"),
        F.col("anio").alias("anio_c"),
        F.col("mes").alias("mes_c"),
        "precip_real_mm"
    ),
    on=[
        dataset.departamento_destino == F.col("dep_c"),
        dataset.anio == F.col("anio_c"),
        dataset.mes_num == F.col("mes_c"),
    ]
).select("precip_pronosticada_mm", "precip_real_mm", "tm_vendidas_total").toPandas()

print("Correlación PRONOSTICADA vs ventas:", datos_real["precip_pronosticada_mm"].corr(datos_real["tm_vendidas_total"]).round(3))
print("Correlación REAL vs ventas:", datos_real["precip_real_mm"].corr(datos_real["tm_vendidas_total"]).round(3))

In [0]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

X = df_encoded.drop(columns=["tm_vendidas_total"])
y = df_encoded["tm_vendidas_total"]

scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")

print("R² en cada una de las 5 particiones:", scores.round(3))
print(f"R² promedio: {scores.mean():.3f}")